In [ ]:
import sys

sys.path.append('..')  

import json

import pandas as pd

from src import anomaly_detection as ad
from src import forecasting as fc


In [ ]:
with open('../data/labels/combined_windows.json') as f:
    labels = json.load(f)

series_list = [
    'ec2_cpu_utilization_24ae8d',  # série original
    'ec2_cpu_utilization_53ea38',
    'ec2_cpu_utilization_5f5533',
    'ec2_cpu_utilization_77c1ca',
    'ec2_cpu_utilization_825cc2',
    'ec2_cpu_utilization_ac20cd',
    'ec2_cpu_utilization_c6585a',
    'ec2_cpu_utilization_fe7f93',
]

In [ ]:
MARGEM = 288 

def avaliar_serie(nome):
    df = pd.read_csv(f'../data/raw/{nome}.csv', parse_dates=['timestamp'])
    df = df.set_index('timestamp').asfreq('5min')

    windows = labels[f'realAWSCloudwatch/{nome}.csv']
    windows = [(pd.Timestamp(inicio), pd.Timestamp(fim)) for inicio, fim in windows]

    if windows:
        primeira_janela_idx = df.index.get_indexer([windows[0][0]], method='nearest')[0]
        corte = max(int(len(df) * 0.3), primeira_janela_idx - MARGEM)
        corte = min(corte, int(len(df) * 0.9))
    else:
        corte = int(len(df) * 0.8)

    treino = df.iloc[:corte].copy()
    teste = df.iloc[corte:].copy()

    dentro_anomalia = pd.Series(False, index=teste.index)
    for inicio, fim in windows:
        dentro_anomalia |= (teste.index >= inicio) & (teste.index <= fim)

    janelas_no_teste = [w for w in windows if w[0] >= teste.index.min()]

    perfil = fc.calcular_perfil_sazonal(treino)
    previsto = fc.prever(perfil, teste, offset=corte)
    residuo = fc.calcular_residuo(teste, previsto)
    sinal = ad.calcular_sinal(residuo)

    if (~dentro_anomalia).sum() <= 30:
        return {'serie': nome.replace('ec2_cpu_utilization_', ''), 'pct_treino': round(corte/len(df)*100, 1),
                'janelas_teste': len(janelas_no_teste), 'n_alarmes': None,
                'precisao': None, 'recall': None, 'f1': None}

    limiar = ad.calcular_limiar(sinal, ~dentro_anomalia)
    deteccoes = ad.detectar(sinal, limiar)
    alarmes = ad.agrupar_alarmes(deteccoes)
    avaliacao = ad.avaliar_por_janela(alarmes, janelas_no_teste) if janelas_no_teste else None

    return {
        'serie': nome.replace('ec2_cpu_utilization_', ''),
        'pct_treino': round(corte / len(df) * 100, 1),
        'janelas_teste': len(janelas_no_teste),
        'n_alarmes': len(alarmes),
        'precisao': round(avaliacao['precisao'], 3) if avaliacao else None,
        'recall': round(avaliacao['recall'], 3) if avaliacao else None,
        'f1': round(avaliacao['f1'], 3) if avaliacao else None,
    }

In [ ]:
resultados = [avaliar_serie(nome) for nome in series_list]
df_resultados = pd.DataFrame(resultados)
df_resultados

In [ ]:
df_resultados[['serie', 'pct_treino', 'f1']].sort_values('pct_treino', ascending=False)